In [9]:
from experiment import run_rag_experiment
from metrics import metrics_summary
from dataclasses import dataclass, asdict
from generate.query import docs_retriver, db_connector
from augment.embed import embed
import pandas as pd

In [3]:
query="When can HMRC issue a discovery assessment?"
query_embedding = embed(query)
TOP_K = 5
results = db_connector(query_embedding, TOP_K)
retrieved_docs = docs_retriver(results)

result = run_rag_experiment(
    query=query,
    retrieved_docs=retrieved_docs,
    generation_model="llama3",
    enabled_layers=None,  # Enable all layers
    )

print(f"\n\n=== EXPERIMENT METRICS RESULT ===\n\n {result.final_metrics}")

Running layer: lexical_support_layer
Running layer: llm_groundedness_layer
Running layer: selfcheck_layer
Running layer: metarag_layer
Running layer: semantic_entropy_layer


=== EXPERIMENT METRICS RESULT ===

 {'support_score': 0.387, 'hallucination_score_avg': 0.613, 'hallucination_score_max': 1.0, 'risk_level': 'high', 'flagged_layers': ['selfcheck_consistency', 'metarag_mutation', 'semantic_entropy'], 'num_layers': 5}


In [5]:
result.layer_results

[LayerResult(layer_name='lexical_support', support_score=0.508, hallucination_score=0.492, status='pass', details={'claims': [{'claim': "According to the retrieved manual context, HMRC can issue a discovery assessment when a taxpayer has not included something in their self-assessment, for example, if it's been omitted or not reported.", 'support_score': 0.4, 'best_section': 'SAM001', 'best_title': 'SAM001 - Glossary of terms', 'matched_terms': ['discovery', 'example', 'been', 'taxpayer', 'not', 'assessment', 'hmrc', 'included']}, {'claim': 'This is stated in [SAM001] SAM001 - Glossary of terms: "Has not been included in the taxpayer’s self assessment, for example in a discovery assessment".', 'support_score': 0.615, 'best_section': 'SAM001', 'best_title': 'SAM001 - Glossary of terms', 'matched_terms': ['discovery', 'included', 'example', 'been', 'taxpayer', 'not', 'assessment', 'self']}], 'claim_count': 2}),
 LayerResult(layer_name='llm_groundedness_judge', support_score=1.0, hallucin

In [6]:
result.baseline_answer


'According to the retrieved manual context, HMRC can issue a discovery assessment when a taxpayer has not included something in their self-assessment, for example, if it\'s been omitted or not reported. This is stated in [SAM001] SAM001 - Glossary of terms: "Has not been included in the taxpayer’s self assessment, for example in a discovery assessment".'

In [10]:
type(result.layer_results)

df = pd.DataFrame([asdict(layer_result) for layer_result in result.layer_results])

df

,layer_name,support_score,hallucination_score,status,details
0,lexical_support,0.508,0.492,pass,{'claims': [{'claim': 'According to the retrie...
1,llm_groundedness_judge,1.000,0.000,supported,"{'status': 'supported', 'support_score': 1.0, ..."
2,selfcheck_consistency,0.167,0.833,flag,{'samples': ['According to the retrieved conte...
3,metarag_mutation,0.000,1.000,flag,{'factoid_scores': [{'claim': 'According to th...
4,semantic_entropy,0.258,0.742,flag,{'samples': ['According to the retrieved manua...
